# Multimodal Generative AI-Based E-Education System

Run cells 1–5 in order on a Google Colab GPU runtime (NVIDIA Tesla T4, 15 GB VRAM) to install dependencies, download model checkpoints, and launch the Gradio Teacher/Student web interface.


In [ ]:
# 1. Verify PyTorch and CUDA availability
import os
import sys
import torch

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Python {sys.version.split()[0]}")


In [ ]:
# 2. Install system tools and Python dependencies
import os
import shutil
import site
import subprocess
import sys

!apt-get update -qq
!apt-get install -y -qq ffmpeg espeak-ng espeak git libportaudio2

!pip install -q --only-binary=:all: \
    Pillow scipy pydub imageio-ffmpeg safetensors accelerate \
    yacs gradio python-docx python-pptx pandas scikit-image \
    tiktoken more-itertools tqdm diffusers transformers \
    onnxruntime pymatting pooch ctransformers edge-tts gTTS \
    addict lmdb pyyaml requests yapf kornia

!pip install -q --no-deps --no-build-isolation rembg openai-whisper future

# Install pure-Python vision restoration dependencies directly from official repositories
!git clone -q --depth 1 https://github.com/rlabbe/filterpy.git /tmp/filterpy_repo 2>/dev/null || true
!git clone -q --depth 1 https://github.com/XPixelGroup/BasicSR.git /tmp/basicsr_repo 2>/dev/null || true
!git clone -q --depth 1 https://github.com/xinntao/facexlib.git /tmp/facexlib_repo 2>/dev/null || true
!git clone -q --depth 1 https://github.com/TencentARC/GFPGAN.git /tmp/gfpgan_repo 2>/dev/null || true

site_dir = site.getsitepackages()[0]
for repo_dir, pkg_name, ver_str in [
    ("/tmp/filterpy_repo", "filterpy", "1.4.5"),
    ("/tmp/basicsr_repo", "basicsr", "1.4.2"),
    ("/tmp/facexlib_repo", "facexlib", "0.3.0"),
    ("/tmp/gfpgan_repo", "gfpgan", "1.4.0"),
]:
    src_pkg = os.path.join(repo_dir, pkg_name)
    dst_pkg = os.path.join(site_dir, pkg_name)
    if os.path.exists(src_pkg):
        if os.path.exists(dst_pkg):
            shutil.rmtree(dst_pkg)
        shutil.copytree(src_pkg, dst_pkg)
        with open(os.path.join(dst_pkg, "version.py"), "w", encoding="utf-8") as vf:
            vf.write(f"__version__ = '{ver_str}'\n__gitsha__ = 'unknown'\nversion_info = ({ver_str.replace('.', ', ')})\n")

# Update deprecated torchvision import in BasicSR
deg_path = os.path.join(site_dir, "basicsr", "data", "degradations.py")
if os.path.exists(deg_path):
    with open(deg_path, "r", encoding="utf-8") as f:
        txt = f.read()
    txt = txt.replace(
        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
        "from torchvision.transforms.functional import rgb_to_grayscale",
    )
    with open(deg_path, "w", encoding="utf-8") as f:
        f.write(txt)

if sys.version_info < (3, 13):
    subprocess.run(["pip", "install", "-q", "coqui-tts==0.24.2"], check=False)

print("Dependencies installed.")


In [ ]:
# 3. Clone SadTalker and download pre-trained checkpoints
!rm -rf /content/SadTalker
!git clone -q https://github.com/OpenTalker/SadTalker.git /content/SadTalker
!cd /content/SadTalker && bash scripts/download_models.sh
!wget -q -nc https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
print("Checkpoints ready.")


In [ ]:
# 4. Clone this repository (or upload src/ to Colab) and apply SadTalker compatibility patches
import sys

if not os.path.exists("/content/Multimodal-E-Education-System"):
    !git clone -q https://github.com/aryansingal823/Multimodal-E-Education-System.git /content/Multimodal-E-Education-System

sys.path.insert(0, "/content/Multimodal-E-Education-System/src")
from sadtalker_patches import patch_sadtalker_repo

patch_sadtalker_repo("/content/SadTalker")
print("SadTalker compatibility patches applied.")


In [ ]:
# 5. Initialize models and launch the Gradio E-Education System interface
from app import create_complete_app, init_models, TMP_DIR, BASE_DATA_DIR, SHARED_FILES_DIR
import os

init_models()
demo = create_complete_app()
demo.launch(
    debug=True,
    share=True,
    allowed_paths=[
        TMP_DIR,
        BASE_DATA_DIR,
        "/tmp",
        os.path.join(SHARED_FILES_DIR, "notes"),
        os.path.join(SHARED_FILES_DIR, "assignments"),
        os.path.join(SHARED_FILES_DIR, "resources"),
    ],
    server_name="0.0.0.0",
    server_port=7860,
)
